# Week 2 Labs: Storage Architectures & Data Organization

Welcome to the Week 2 Lab! By the end of this session, you will understand the importance of file formats, learn how to navigate a data swamp, and grasp the core mechanics of Delta Lake—all using pure Python on your local machine.

## Lab 1: Surviving the Local Data Swamp

**Objective:** Understand the reality of raw storage and the transition to a governed landing zone.

Let's generate a messy data swamp directory.

In [ ]:
import pandas as pd
import json
import os

# Create swamp directory
swamp_dir = 'week2/raw_swamp'
os.makedirs(swamp_dir, exist_ok=True)

# 1. Valid CSV
pd.DataFrame({'id': [1,2], 'val': ['A','B']}).to_csv(f'{swamp_dir}/data1.csv', index=False)
# 2. Valid CSV
pd.DataFrame({'id': [3,4], 'val': ['C','D']}).to_csv(f'{swamp_dir}/data2.csv', index=False)
# 3. Random text file
with open(f'{swamp_dir}/readme.txt', 'w') as f: f.write('This is not data.')
# 4. Valid JSON
with open(f'{swamp_dir}/valid.json', 'w') as f: json.dump({'sensor': 1, 'reading': 42}, f)
# 5. Invalid JSON
with open(f'{swamp_dir}/invalid.json', 'w') as f: f.write('{sensor: 1, reading: 42') # missing quotes

print('Created /week2/raw_swamp with mixed, messy files.')

### Task 1.1: Exploration & Failure
Try looping through all files in `raw_swamp` and reading them all as CSVs. Watch it fail.

In [ ]:
import glob
import pandas as pd

# Try putting all files from 'week2/raw_swamp/' into a single DataFrame using pd.read_csv()
# Don't try to fix the error yet, just see what happens.
all_files = glob.glob('week2/raw_swamp/*')

try:
    combined_df = pd.concat((pd.read_csv(f) for f in all_files))
    print("Success!")
except Exception as e:
    print(f"FAILED!\nError Type: {type(e).__name__}\nError Message: {e}")

### Task 1.2 & 1.3: Swamp Cleanup and Validation

In real-world data lakes, the raw landing zone (Bronze layer) often accumulates broken or unexpected files. Before data analysts can use it, data engineers must build pipelines to filter the good from the bad. 

Here, we will simulate an automated pipeline that iterates through the `raw_swamp`. It will validate file types and check if JSON files are properly formatted:
- **Valid data** is promoted to the `bronze_validated` layer (to become "Silver", we would need to enforce schema and write to Parquet/Delta, which we cover later!).
- **Corrupt or unknown data** is isolated into a `quarantine` folder for later inspection, keeping our main pipeline safe from crashes.

In [ ]:
import shutil
import glob
import json
import os

bronze_dir = 'week2/bronze_validated'
quarantine_dir = 'week2/quarantine'

# Create directories if they don't exist
os.makedirs(bronze_dir, exist_ok=True)
os.makedirs(quarantine_dir, exist_ok=True)

# Loop through files in raw_swamp
for file_path in glob.glob('week2/raw_swamp/*'):
    filename = os.path.basename(file_path)
    
    if filename.endswith('.csv'):
        # If it's a .csv, move it to bronze_validated
        shutil.move(file_path, os.path.join(bronze_dir, filename))
        print(f"Moved CSV {filename} to {bronze_dir}")
    elif filename.endswith('.json'):
        # If it's a .json, attempt to parse it with json.load()
        try:
            with open(file_path, 'r') as f:
                json.load(f)
            # If it works -> bronze_validated
            shutil.move(file_path, os.path.join(bronze_dir, filename))
            print(f"Moved valid JSON {filename} to {bronze_dir}")
        except json.JSONDecodeError:
            # If Error -> quarantine
            shutil.move(file_path, os.path.join(quarantine_dir, filename))
            print(f"quarantined invalid JSON {filename}")
    else:
        # Any other extensions (like .txt) go to quarantine.
        shutil.move(file_path, os.path.join(quarantine_dir, filename))
        print(f"quarantined unknown file format {filename}")

print("\nCleanup complete!")

## Lab 2: The Format Showdown (Pure Python)

**Objective:** Prove why Columnar Parquet is essential for analytical speed and storage efficiency.

First, let's generate a large dummy CSV file (~100-200MB depending on your system).

In [ ]:
!pip install pyarrow
import pandas as pd
import numpy as np
import os
import time

print('Generating dummy CSV data...')
num_rows = 5_000_000 # Generating 5 million rows to reliably demonstrate performance differences
df = pd.DataFrame({
    'transaction_id': np.arange(num_rows),
    'user_id': np.random.randint(1, 100000, size=num_rows),
    'amount': np.random.random(size=num_rows) * 1000,
    'status': np.random.choice(['success', 'failed', 'pending'], size=num_rows),
    'timestamp': pd.date_range(start='1/1/2021', periods=num_rows, freq='s')
})

df.to_csv('week2/transactions_large.csv', index=False)
print(f'Generated transactions_large.csv - Size: {os.path.getsize("week2/transactions_large.csv") / (1024*1024):.2f} MB')

### Task 2.1: Format Conversion
Convert the CSV to Parquet using Pandas (`to_parquet()`).

In [ ]:
import pandas as pd
# Read 'week2/transactions_large.csv' into a pandas DataFrame
df_csv = pd.read_csv('week2/transactions_large.csv')

# Save the dataframe back to disk as 'week2/transactions_large.parquet'
df_csv.to_parquet('week2/transactions_large.parquet', index=False)
print("Conversion complete.")

### Task 2.2: Size Comparison
Compare the file sizes on disk.

In [ ]:
import os
# Use os.path.getsize() to print out the size of both files in MB.
# Observe the difference!
csv_size = os.path.getsize('week2/transactions_large.csv') / (1024*1024)
pq_size = os.path.getsize('week2/transactions_large.parquet') / (1024*1024)

print(f"CSV Size: {csv_size:.2f} MB")
print(f"Parquet Size: {pq_size:.2f} MB")
print(f"Space saved: {csv_size - pq_size:.2f} MB ({(1 - pq_size/csv_size)*100:.1f}%)")

### Task 2.3: Speed Test
Measure how long it takes to calculate the total sum of the 'amount' column using the CSV vs the Parquet file.

**Note:** You might ask, "Why not use `pd.read_csv('...', usecols=['amount'])` to make it fair?" Even if you tell Pandas to only use one column from a CSV, the underlying physical disk I/O still has to scan the *entire* row-based file structure on disk to find the values. Parquet allows the disk reader to skip directly to the contiguous block of 'amount' data, which is the core superpower of columnar storage!

In [ ]:
import time
import pandas as pd
# Use time.time() to measure reading the CSV and calculating df['amount'].sum()
start_csv = time.time()
csv_df = pd.read_csv('week2/transactions_large.csv')
csv_sum = csv_df['amount'].sum()
csv_time = time.time() - start_csv
print(f"CSV read and sum time: {csv_time:.4f} seconds (Sum: {csv_sum:.2f})")

# Use time.time() to measure reading ONLY the 'amount' column from the Parquet file and calculating sum()
# Hint: pd.read_parquet(..., columns=['amount'])
start_pq = time.time()
pq_df = pd.read_parquet('week2/transactions_large.parquet', columns=['amount'])
pq_sum = pq_df['amount'].sum()
pq_time = time.time() - start_pq
print(f"Parquet read and sum time: {pq_time:.4f} seconds (Sum: {pq_sum:.2f})")

print(f"Parquet was {csv_time / pq_time:.1f}x faster!")

## Lab 3: Anatomy of a Delta Table (Without Spark)

**Objective:** Bridge the gap to the Lakehouse by understanding ACID transactions locally using `deltalake` (delta-rs).

### Task 3.1: Table Creation
Take a pandas dataframe and write it as a Delta table using `write_deltalake`.

In [ ]:
!pip install deltalake
import pandas as pd
from deltalake import write_deltalake, DeltaTable

# Make a simple cleaned dataframe
clean_df = pd.DataFrame({'user': [1,2,3], 'score': [100, 95, 80]})

# Use write_deltalake to save clean_df to a local folder named 'week2/my_local_delta_table'
write_deltalake('week2/my_local_delta_table', clean_df, mode='overwrite')
print("Delta table successfully created.")

### Task 3.2: Under the Hood
**STOP!** Before you write more code, open your computer's file explorer. Go to the folder where this notebook lives, and open `my_local_delta_table`. 

Look inside the `_delta_log/` folder. Open the `.json` file and inspect it! You are seeing the actual transaction log of Delta Lake.

### Task 3.3: Overwrite and Time Travel
Let's overwrite the table with bad data, then use versioning to fix it!

In [ ]:
import pandas as pd
from deltalake import write_deltalake, DeltaTable

# OVERWRITE WITH BAD DATA (Sabotage!)
bad_df = pd.DataFrame({'user': [999], 'score': [-50]})
write_deltalake('week2/my_local_delta_table', bad_df, mode='overwrite')

print('Current Delta Table Data:')
print(DeltaTable('week2/my_local_delta_table').to_pandas())

In [ ]:
from deltalake import DeltaTable

# Use DeltaTable() to read the table again, but pass `version=0` as an argument.
dt_v0 = DeltaTable('week2/my_local_delta_table', version=0)

# Print the resulting pandas dataframe to prove you rescued the original data.
print("Rescued Original Data (Version 0):")
print(dt_v0.to_pandas())